<details><summary style="display:list-item; font-size:16px; color:blue;">Jupyter Help</summary>
    
Having trouble testing your work? Double-check that you have followed the steps below to write, run, save, and test your code!

[Click here for a walkthrough GIF of the steps below](https://static-assets.codecademy.com/Courses/ds-python/jupyter-help.gif)

Run all initial cells to import libraries and datasets. Then follow these steps for each question:

1. Add your solution to the cell with `## YOUR SOLUTION HERE ##`.
2. Run the cell by selecting the `Run` button or the `Shift`+`Enter` keys.
3. Save your work by selecting the `Save` button, the `command`+`s` keys (Mac), or `control`+`s` keys (Windows).
4. Select the `Test Work` button at the bottom left to test your work.

![Screenshot of the buttons at the top of a Jupyter Notebook. The Run and Save buttons are highlighted](https://static-assets.codecademy.com/Paths/ds-python/jupyter-buttons.png)

#### Setup Cell

Run the setup cell below to import required libraries, initialize the language model, and define the shared state.

`MemorySaver` stores graph state in memory to enable pause and resume, while `interrupt()` pauses execution for human review and `Command` resumes it with the human's decision.

The language model `llm` is initialized using `gpt-4o` with a temperature of `0.5`. A `ReviewState` schema and a `publish_content` function are also defined in the setup.

**Note:** The OpenAI API key is already configured in this workspace, so no additional setup is needed. When working in an external environment, refer to the [OpenAI API](https://platform.openai.com/docs/quickstart?language=python) documentation for details on configuring your API key.

In [ ]:
# Third-party imports
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from typing import Literal
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
import warnings

warnings.filterwarnings("ignore")

# Initialize the language model
llm = ChatOpenAI(model="gpt-4o", temperature=0.5)

# Define the state schema for the pipeline
class ReviewState(TypedDict):
    topic: str
    draft: str
    feedback: str
    status: str

# Define a node to mark the status as published
def publish_content(state: ReviewState):
    print("Draft published.")
    return {"status": "published"}

#### Checkpoint 1/5: Define the Draft and Revise Nodes

Let's create two functions that accept a `state` parameter of type `ReviewState` to generate the initial draft and revise it based on human feedback.

- **Drafting:** Define a function `write_draft` that uses the `llm` to write a short blog post (2–3 paragraphs) based on the `topic`, and returns a dictionary with a `draft` key containing the generated content.

- **Revising:** Define a function `revise_content` that uses the `llm` to revise the `draft` based on the `feedback`, and returns a dictionary with a `draft` key containing the revised content.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Define a node that generates an initial draft
def write_draft(state: ReviewState):
    result = llm.invoke(
        f"Write a short blog post (2-3 paragraphs) on: {state['topic']}"
    )
    print("Draft created.")
    return {"draft": result.content}

# Define a node that revises the draft based on feedback
def revise_content(state: ReviewState):
    result = llm.invoke(
        f"Revise this blog post based on the feedback.\n\n"
        f"Draft:\n{state['draft']}\n\n"
        f"Feedback: {state['feedback']}\n"
    )
    print("Draft revised.")
    return {"draft": result.content}

#### Checkpoint 2/5: Define the Editor Review Node

With the draft and revise nodes in place, let's set up memory persistence and complete the function that pauses execution for human review.

- Create a `MemorySaver` instance and assign it to `cp`.

- **Reviewing:** Complete the `editor_review` function.

   - Use `interrupt()` to pause execution by passing a dictionary with the `draft` and an instruction to reply with `"approve"` or `"revise: <feedback>"`. Store the response in `decision`.

   - If the decision starts with `"approve"`, return a `Command` with `goto` set to `"publish"`; otherwise, return a `Command` with `goto` set to `"revise"` and `update` containing the extracted `feedback`.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Initialize an in-memory checkpointer to save graph state
cp = MemorySaver()

# Define a node that pauses for review and routes based on the decision
def editor_review(state: ReviewState) -> Command[Literal["publish", "revise"]]:
    """Pause for editor review and route based on decision."""
    decision = interrupt({
        "draft": state["draft"],
        "instruction": "Review this draft. Reply: 'approve' or 'revise: <feedback>'"
    })
    if decision.startswith("approve"):
        return Command(goto="publish")
    else:
        feedback = decision.replace("revise:", "").strip()
        return Command(goto="revise", update={"feedback": feedback})

#### Checkpoint 3/5: Build the Graph Structure

With all node functions ready, let's construct the graph by adding them as nodes and defining the execution flow.

- Create a `StateGraph` instance with `ReviewState` as its state type and assign it to `builder`. 

- Add four nodes: `write` (using `write_draft`), `review` (using `editor_review`), `publish` (using `publish_content`), and `revise` (using `revise_content`). 

- Wire the execution flow with an edge from `START` to `write`, edges from `write` and `revise` to `review`, and an edge from `publish` to `END`.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Initialize the graph with the defined state schema
builder = StateGraph(ReviewState)

# Add nodes to the graph
builder.add_node("write", write_draft)
builder.add_node("review", editor_review)
builder.add_node("publish", publish_content)
builder.add_node("revise", revise_content)

# Connect the nodes using edges
builder.add_edge(START, "write")
builder.add_edge("write", "review")
builder.add_edge("revise", "review")
builder.add_edge("publish", END)

#### Checkpoint 4/5: Compile and Run the Graph with Interruption

Now let's compile the graph with memory persistence and run it until it pauses for human review.

- Call `compile()` on `builder` with `checkpointer` set to `cp`, and assign the result to `editorial_graph`. 

- Invoke it with a state dictionary containing the `topic` variable, empty strings for `draft`, `feedback`, and `status`, along with the `config` dictionary. Store the output in `result`.

- Uncomment the `print()` statements to view the draft and review instructions when the graph pauses.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Compile the graph with the checkpointer
editorial_graph = builder.compile(checkpointer=cp)

topic = "How AI tools can reduce repetitive work?"

## Config is set to track execution using a unique thread_id
config ={"configurable": {"thread_id": "article-001"}}

# Invoke the pipeline with the initial state
result = editorial_graph.invoke(
    {"topic": topic,
     "draft": "", "feedback": "", "status": ""},
    config=config
)

print("\nGraph paused at editor review.")
if "__interrupt__" in result:
    payload = result["__interrupt__"][0].value
    print(f"Instruction: {payload['instruction']}")
    print(f"\nDraft for review:\n{payload['draft'][:500]}...")

#### Checkpoint 5/5: Resume the Graph with Human Feedback

Now let's resume the graph by providing editor feedback for revision and then approving the final draft.

- **Revise:** Invoke `editorial_graph` with a `Command` whose `resume` is set to `"revise: Add examples of where AI can safely reduce repetitive work"`, along with the `config` parameter. Store the result in `revised`.

- **Approve:** Invoke `editorial_graph` again with a `Command` whose `resume` is set to `"approve"`, along with the `config` parameter. Store the result in `final`.

- Uncomment the `print()` statements to view the revised draft and the final status after approval.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Resume the graph with a revision request
revised = editorial_graph.invoke(
    Command(resume="revise: Add examples of where AI can safely reduce repetitive work"),
    config=config
)
print("Graph paused again after revision.")
if "__interrupt__" in revised:
    print(f"Revised draft preview:\n{revised['__interrupt__'][0].value['draft'][:500]}...")

print("\n" + "-" * 100)

# Resume the graph with an approval decision
final = editorial_graph.invoke(
    Command(resume="approve"),
    config=config
)
print(f"\nStatus: {final['status']}")

<details> 
    
<summary><strong>Summary</strong></summary> 

In this exercise, human-in-the-loop is used during execution. `interrupt()` pauses the graph and shows the draft, `MemorySaver` keeps the state during review, and `Command(resume=...)` resumes the graph with the decision.

</details>